# Naz Lab Official Colab Launcher

Runs the official single Streamlit dashboard at `master_dashboard/app_official.py` on port `8502`.

This notebook replaces the old Markdown-only launcher for normal Colab use.

In [ ]:
# 1. Mount Drive and prepare folders
from pathlib import Path
from google.colab import drive
import os, time

drive.mount('/content/drive', force_remount=False)
BASE = Path('/content/drive/MyDrive/NazLab')
folders = [
    BASE / 'models', BASE / 'models' / 'ollama',
    BASE / 'text_outputs', BASE / 'chat_outputs', BASE / 'script_outputs', BASE / 'image_prompts',
    BASE / 'image_outputs', BASE / 'voice_outputs', BASE / 'voice_outputs' / 'reference_audio',
    BASE / 'video_outputs', BASE / 'job_queue', BASE / 'job_queue' / 'image_jobs',
    BASE / 'job_queue' / 'voice_jobs', BASE / 'job_queue' / 'video_jobs',
    BASE / 'social_review', BASE / 'final_packages', BASE / 'final_packages' / 'exports',
    BASE / 'reference_images', BASE / 'config', BASE / 'logs',
]
def safe_mkdir(path: Path):
    try:
        if path.exists() or path.is_symlink():
            print('OK exists:', path)
            return
        path.mkdir(parents=True, exist_ok=True)
        print('Created:', path)
    except FileExistsError:
        print('OK already exists or Drive race tolerated:', path)
    except Exception as exc:
        print('WARNING:', path, type(exc).__name__, exc)
for folder in folders:
    safe_mkdir(folder)
os.environ['OLLAMA_MODELS'] = str(BASE / 'models' / 'ollama')
print('Drive ready:', BASE)

In [ ]:
# 2. Download latest main branch
import shutil, zipfile, subprocess, sys
from pathlib import Path

REPO_ZIP_URL = 'https://github.com/nazmul73/naz-lab/archive/refs/heads/main.zip'
REPO_DIR = Path('/content/naz-lab')
ZIP_PATH = Path('/content/naz-lab-main.zip')
EXTRACTED_DIR = Path('/content/naz-lab-main')

for path in [REPO_DIR, EXTRACTED_DIR]:
    if path.exists():
        shutil.rmtree(path)
if ZIP_PATH.exists():
    ZIP_PATH.unlink()
subprocess.run(['wget', '-q', '-O', str(ZIP_PATH), REPO_ZIP_URL], check=True)
with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
    zf.extractall('/content')
EXTRACTED_DIR.rename(REPO_DIR)
print('Repo ready:', REPO_DIR)

In [ ]:
# 3. Install Python requirements
import subprocess, sys
from pathlib import Path
REQ = Path('/content/naz-lab/requirements.txt')
if REQ.exists():
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REQ)], check=False)
else:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'streamlit', 'requests', 'Pillow', 'watchdog'], check=False)
print('Requirements ready')

In [ ]:
# 4. Optional Ollama install/start helper
import os, shutil, subprocess, time
from pathlib import Path

os.environ['OLLAMA_MODELS'] = '/content/drive/MyDrive/NazLab/models/ollama'
ollama_bin = shutil.which('ollama') or '/usr/local/bin/ollama'
if not Path(ollama_bin).exists():
    print('Ollama not found. Installing...')
    subprocess.run('curl -fsSL https://ollama.com/install.sh | sh', shell=True, check=False)
    ollama_bin = shutil.which('ollama') or '/usr/local/bin/ollama'
if Path(ollama_bin).exists():
    subprocess.run("pkill -f 'ollama serve' || true", shell=True, check=False)
    log = open('/content/ollama_naz_lab.log', 'w')
    subprocess.Popen([ollama_bin, 'serve'], stdout=log, stderr=log, env=os.environ.copy())
    time.sleep(5)
    subprocess.run([ollama_bin, 'pull', 'gemma2:2b'], check=False)
    subprocess.run([ollama_bin, 'pull', 'qwen2.5:1.5b'], check=False)
else:
    print('WARNING: Ollama binary still not found. Dashboard can open, but model generation may fallback.')

In [ ]:
# 5. Smoke check and py_compile validation
import subprocess, sys
from pathlib import Path
REPO = Path('/content/naz-lab')
SMOKE = REPO / 'scripts' / 'backend_smoke_check.py'
if SMOKE.exists():
    subprocess.run([sys.executable, str(SMOKE)], check=False)
targets = [
    REPO / 'master_dashboard' / 'naz_lab_dashboard_v12.py',
    REPO / 'master_dashboard' / 'naz_lab_nav.py',
    REPO / 'master_dashboard' / 'naz_lab_text_panel.py',
    REPO / 'master_dashboard' / 'naz_lab_voice_panel.py',
    REPO / 'master_dashboard' / 'naz_lab_image_panel.py',
    REPO / 'master_dashboard' / 'naz_lab_review_panel.py',
    REPO / 'master_dashboard' / 'naz_lab_facebook_panel.py',
    REPO / 'master_dashboard' / 'app_main.py',
    REPO / 'master_dashboard' / 'app_official.py',
    REPO / 'text_workstation' / 'app_phase110.py',
    REPO / 'voice_workstation' / 'voice_backend.py',
    REPO / 'social_agent' / 'facebook_graph_backend.py',
]
for target in targets:
    if target.exists():
        r = subprocess.run([sys.executable, '-m', 'py_compile', str(target)], capture_output=True, text=True)
        print(('PASS' if r.returncode == 0 else 'FAIL'), target.relative_to(REPO))
        if r.returncode != 0:
            print(r.stderr)
            raise RuntimeError(target)
print('Validation ready')

In [ ]:
# 6. Launch official dashboard on one Streamlit process
import subprocess, sys, time, os
from pathlib import Path
from google.colab import output

REPO = Path('/content/naz-lab')
APP = REPO / 'master_dashboard' / 'app_official.py'
PORT = 8502
LOG = Path('/content/streamlit_naz_lab_runtime.log')
subprocess.run('pkill -f streamlit || true', shell=True, check=False)
cmd = [
    sys.executable, '-m', 'streamlit', 'run', str(APP),
    '--server.port', str(PORT),
    '--server.address', '0.0.0.0',
    '--server.headless', 'true',
    '--server.enableCORS', 'false',
    '--server.enableXsrfProtection', 'false',
]
with open(LOG, 'w') as log_file:
    subprocess.Popen(cmd, stdout=log_file, stderr=log_file, cwd=str(REPO / 'master_dashboard'), env=os.environ.copy())
time.sleep(6)
print('NAZ LAB RUNTIME READY')
print('Opening Colab proxy window on port', PORT)
print('Log file:', LOG)
output.serve_kernel_port_as_window(PORT)